# Exploring the raw feed — column by column, then payload by payload

**Goal**: build a mental model of the two parquet files before deciding anything.

**Plan**:
1. Land in the data — dimensions, first rows.
2. Walk the outer (delivery-metadata) columns one group at a time. What does each mean? How does it vary?
3. Open the `payload_json` string — it's the real Open Finance API response. Explore per `(investment_type, payload_kind)` shape.
4. Same for `transaction_json`.
5. End with a scratch-pad of observations to feed into the design docs.

**Spec reference** (keep open in another tab):
- Portal: https://openfinancebrasil.atlassian.net/wiki/spaces/OF/overview
- Investments Swagger UI: https://openbanking-brasil.github.io/openapi/swagger-apis/investments/?urls.primaryName=1.0.1
- Raw YAMLs on GitHub: https://github.com/OpenBanking-Brasil/openapi/tree/main/swagger-apis

## 1 · Landing

In [1]:
import duckdb, json, pandas as pd
from pathlib import Path

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 40)

DATA = Path('../data/raw')
con = duckdb.connect()
con.execute(f"CREATE VIEW pos AS SELECT * FROM read_parquet('{DATA/'raw_positions.parquet'}')")
con.execute(f"CREATE VIEW txn AS SELECT * FROM read_parquet('{DATA/'raw_transactions.parquet'}')")

print(f'positions:    {con.execute("SELECT count(*) FROM pos").fetchone()[0]:>8,} rows')
print(f'transactions: {con.execute("SELECT count(*) FROM txn").fetchone()[0]:>8,} rows')

positions:     210,647 rows
transactions:  321,117 rows


### The two schemas

Everything is `VARCHAR` — even timestamps and identifiers. Typing is deferred to the staging layer.

Notice: positions has `payload_kind` and `payload_json`; transactions has `payload_source`, `page`, `query_window`, and `transaction_json`.

In [2]:
print('--- POSITIONS ---'); print(con.execute('DESCRIBE pos').df().to_string(index=False))
print('\n--- TRANSACTIONS ---'); print(con.execute('DESCRIBE txn').df().to_string(index=False))

--- POSITIONS ---
        column_name column_type null  key default extra
     institution_id     VARCHAR  YES None    None  None
   institution_name     VARCHAR  YES None    None  None
           party_id     VARCHAR  YES None    None  None
         account_id     VARCHAR  YES None    None  None
      connection_id     VARCHAR  YES None    None  None
        snapshot_id     VARCHAR  YES None    None  None
snapshot_created_at     VARCHAR  YES None    None  None
      investment_id     VARCHAR  YES None    None  None
    investment_type     VARCHAR  YES None    None  None
       payload_kind     VARCHAR  YES None    None  None
             s3_uri     VARCHAR  YES None    None  None
       payload_json     VARCHAR  YES None    None  None
        ingested_at     VARCHAR  YES None    None  None

--- TRANSACTIONS ---
        column_name column_type null  key default extra
     institution_id     VARCHAR  YES None    None  None
   institution_name     VARCHAR  YES None    None  None
        

### Eyeball a few raw rows

The JSON columns will look like giant strings. That's fine — we'll open them below.

In [3]:
con.execute('SELECT * FROM pos LIMIT 3').df()

,institution_id,institution_name,party_id,account_id,connection_id,snapshot_id,snapshot_created_at,investment_id,investment_type,payload_kind,s3_uri,payload_json,ingested_at
0,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,9e246f99-6d8d-5e76-aab3-ee80ec0dcf50,2026-07-30T09:00:01Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,detail,s3://of-snapshots-sample/prod/9e246f99-6d8d-5e76-aab3-ee80ec0dcf50/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""data"":{""isinCode"":""BRSTNCLTN8C3"",""productName"":""Tesouro Prefixado 2027"",""remuneration"":{""indexer"":""PRE_FIXADO"",""ra...",2026-07-30T09:01:01Z
1,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,9e246f99-6d8d-5e76-aab3-ee80ec0dcf50,2026-07-30T09:00:01Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,balances,s3://of-snapshots-sample/prod/9e246f99-6d8d-5e76-aab3-ee80ec0dcf50/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""data"":{""referenceDateTime"":""2026-07-30T09:00:00Z"",""updatedUnitPrice"":{""amount"":""504.63"",""currency"":""BRL""},""grossAm...",2026-07-30T09:01:01Z
2,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,9e246f99-6d8d-5e76-aab3-ee80ec0dcf50,2026-07-30T09:00:01Z,be6dd9c2-e4ca-518b-9966-5e2fc451ad72,CREDIT_FIXED_INCOMES,detail,s3://of-snapshots-sample/prod/9e246f99-6d8d-5e76-aab3-ee80ec0dcf50/investments/credit-fixed-incomes/be6dd9c2-e4ca-51...,"{""data"":{""issuerInstitutionCnpjNumber"":""92894922000108.00"",""isinCode"":""BRDEBSEC01A1"",""investmentType"":""DEBENTURES"",""...",2026-07-30T09:01:01Z


In [4]:
con.execute('SELECT * FROM txn LIMIT 3').df()

,institution_id,institution_name,party_id,account_id,connection_id,snapshot_id,snapshot_created_at,investment_id,investment_type,payload_source,page,s3_uri,transaction_json,query_window,ingested_at
0,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,9e246f99-6d8d-5e76-aab3-ee80ec0dcf50,2026-07-30T09:00:01Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,transactions,1,s3://of-snapshots-sample/prod/9e246f99-6d8d-5e76-aab3-ee80ec0dcf50/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""type"":""ENTRADA"",""transactionType"":""COMPRA"",""transactionDate"":""2026-03-12"",""transactionUnitPrice"":{""amount"":""500.00...",2025-07-30/2026-07-30,2026-07-30T09:01:01Z
1,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,42fef1b2-7a82-5017-92aa-67662d678121,2026-08-02T09:00:02Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,transactions,1,s3://of-snapshots-sample/prod/42fef1b2-7a82-5017-92aa-67662d678121/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""type"":""ENTRADA"",""transactionType"":""COMPRA"",""transactionDate"":""2026-03-12"",""transactionUnitPrice"":{""amount"":""500.00...",2025-08-02/2026-08-02,2026-08-02T09:06:02Z
2,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,35b95095-d1b1-5252-9476-10b4c09b2f44,2026-08-05T09:00:03Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,transactions,1,s3://of-snapshots-sample/prod/35b95095-d1b1-5252-9476-10b4c09b2f44/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""type"":""ENTRADA"",""transactionType"":""COMPRA"",""transactionDate"":""2026-03-12"",""transactionUnitPrice"":{""amount"":""500.00...",2025-08-05/2026-08-05,2026-08-05T09:07:03Z


## 2 · Outer columns — delivery metadata

The 15 outer columns are metadata that Decade's ingestion layer wraps around the raw provider payloads. Understanding them tells us *how* records arrive, before we care what's inside them.

We'll walk four groups:
- **Who** — the institution and the customer
- **Where** — account, connection
- **When** — snapshot & ingestion timestamps
- **What** — investment identifiers + which endpoint the payload came from

### 2.1 · Who — `institution_id`, `institution_name`, `party_id`

- **`institution_id`** — Decade's identifier for the source bank. Should map 1:1 with `institution_name`.
- **`institution_name`** — human-readable. Not authoritative; can drift.
- **`party_id`** — the customer. "Party" is Open Finance vocabulary (`PF` = natural person, `PJ` = legal entity).

Watch for: multiple `institution_name`s per `institution_id` (rebrand, typo), or vice-versa (bad joins upstream).

In [5]:
# Institutions and their volume
con.execute("""
  SELECT institution_id, institution_name,
         count(*) AS positions_rows,
         count(DISTINCT party_id) AS customers
  FROM pos
  GROUP BY 1,2
  ORDER BY positions_rows DESC
""").df()

,institution_id,institution_name,positions_rows,customers
0,00000000000001,Nubank,80567,160
1,00000000000003,Itau,39314,65
2,00000000000002,Banco XP S.A.,28558,79
3,00000000000004,BTG Banking,25468,62
4,00000000000006,C6 Bank,15828,36
5,00000000000005,Banco Inter PF,14536,42
6,00000000000007,PicPay,4118,22
7,00000000000008,Banco do Brasil,2258,17


In [6]:
# Are institution_id and institution_name in bijection? (they should be)
con.execute("""
  SELECT institution_id, count(DISTINCT institution_name) AS n_names
  FROM pos GROUP BY 1 HAVING count(DISTINCT institution_name) > 1
""").df()

,institution_id,n_names


In [7]:
# Customer scale — 'a few hundred synthetic customers'
con.execute("SELECT count(DISTINCT party_id) AS customers FROM pos").df()

,customers
0,400


### 2.2 · Where — `account_id`, `connection_id`

- **`account_id`** — the investment account within the institution. A customer can hold accounts at multiple institutions and multiple accounts at one institution.
- **`connection_id`** — Decade's identifier for the customer's *authorized connection* to that institution (an Open Finance consent). One consent can produce many snapshots over time.

Cardinality expectation: `customers ≤ connections ≤ accounts`.

In [8]:
con.execute("""
  SELECT
    count(DISTINCT party_id)                  AS customers,
    count(DISTINCT connection_id)             AS connections,
    count(DISTINCT account_id)                AS accounts,
    count(DISTINCT (party_id, institution_id)) AS customer_institution_pairs
  FROM pos
""").df()

,customers,connections,accounts,customer_institution_pairs
0,400,500,500,483


In [9]:
# How many accounts does an average customer have?
con.execute("""
  SELECT accounts_per_customer, count(*) AS n_customers
  FROM (SELECT party_id, count(DISTINCT account_id) AS accounts_per_customer FROM pos GROUP BY 1)
  GROUP BY 1 ORDER BY 1
""").df()

,accounts_per_customer,n_customers
0,1,316
1,2,71
2,3,10
3,4,3


### 2.3 · When — `snapshot_created_at`, `ingested_at`, `query_window` (txn only)

- **`snapshot_created_at`** — when the provider's *snapshot* was constructed. The natural per-snapshot event time.
- **`snapshot_id`** — groups all records that came together in one sync from one institution.
- **`ingested_at`** — when Decade's pipeline received/wrote the record. **This is the natural watermark for incremental jobs** (`arrival_time` in case-study parlance).
- **`query_window`** (transactions only) — the date range the provider was asked for. Transactions endpoints in Open Finance are queried with a `fromTransactionDate` / `toTransactionDate`; this records what we asked.

In [10]:
con.execute("""
  SELECT
    min(snapshot_created_at) AS snapshot_earliest,
    max(snapshot_created_at) AS snapshot_latest,
    min(ingested_at)         AS ingested_earliest,
    max(ingested_at)         AS ingested_latest
  FROM pos
""").df()

,snapshot_earliest,snapshot_latest,ingested_earliest,ingested_latest
0,2026-07-28T09:00:32Z,2026-08-21T10:30:00Z,2026-07-28T09:05:45Z,2026-08-21T10:36:59Z


In [11]:
# Snapshot cadence per institution — the case study says 'irregular and differs by institution'
con.execute("""
  SELECT institution_name, count(DISTINCT snapshot_id) AS snapshots,
         min(snapshot_created_at) AS first_snap, max(snapshot_created_at) AS last_snap
  FROM pos
  GROUP BY 1 ORDER BY snapshots DESC
""").df()

,institution_name,snapshots,first_snap,last_snap
0,Nubank,2152,2026-07-28T09:06:32Z,2026-08-21T10:29:43Z
1,Itau,853,2026-07-28T09:05:38Z,2026-08-21T10:25:01Z
2,Banco XP S.A.,822,2026-07-28T09:04:45Z,2026-08-21T10:25:54Z
3,Banco Inter PF,719,2026-07-28T09:12:42Z,2026-08-21T10:29:19Z
4,BTG Banking,579,2026-07-28T09:00:32Z,2026-08-21T10:19:22Z
5,C6 Bank,254,2026-07-28T09:03:27Z,2026-08-21T10:30:00Z
6,PicPay,114,2026-07-28T09:01:08Z,2026-08-21T10:16:48Z
7,Banco do Brasil,93,2026-07-28T09:03:39Z,2026-08-21T10:26:01Z


In [12]:
# Gap between provider snapshot time and Decade ingest time — how 'late' are records?
con.execute("""
  SELECT institution_name,
         percentile_cont(0.5) WITHIN GROUP (ORDER BY epoch(cast(ingested_at as timestamp)) - epoch(cast(snapshot_created_at as timestamp)))/60 AS median_lag_min,
         percentile_cont(0.95) WITHIN GROUP (ORDER BY epoch(cast(ingested_at as timestamp)) - epoch(cast(snapshot_created_at as timestamp)))/60 AS p95_lag_min
  FROM pos GROUP BY 1 ORDER BY median_lag_min DESC
""").df()

,institution_name,median_lag_min,p95_lag_min
0,Banco do Brasil,6.0,9.0
1,Banco Inter PF,5.0,9.0
2,Itau,5.0,9.0
3,Nubank,5.0,9.0
4,C6 Bank,5.0,9.0
5,PicPay,5.0,9.0
6,BTG Banking,5.0,9.0
7,Banco XP S.A.,5.0,9.0


### 2.4 · What — `investment_id`, `investment_type`, `payload_kind`

- **`investment_id`** — the provider's identifier for one holding. **This is the identifier the case study warns can churn.** The Open Finance spec (`variable-incomes/1.3.0.yml`) *mandates* reuse after 12-month dormancy, but institutions may not comply.
- **`investment_type`** — which product family. Maps 1:1 to the 5 sub-APIs of Open Finance Investments.
- **`payload_kind`** (positions only) — which endpoint the payload came from:
  - `balances` → `/investments/{investmentId}/balances` — the snapshot values (qty, market value)
  - `detail`   → `/investments/{investmentId}` — the security's static identity (ISIN, ticker, CNPJ, dueDate…)

**Modeling implication**: for each holding at each snapshot we receive **two records** (balance + detail). To get a single logical "position row" we must join them by `(institution_id, party_id, account_id, snapshot_id, investment_id)`. Detail can be considered slowly-changing (dueDate, ISIN don't move); balance changes every snapshot.

In [13]:
con.execute("""
  SELECT investment_type, payload_kind, count(*) AS records,
         count(DISTINCT investment_id) AS holdings
  FROM pos GROUP BY 1,2 ORDER BY 1,2
""").df()

,investment_type,payload_kind,records,holdings
0,BANK_FIXED_INCOMES,balances,66183,6303
1,BANK_FIXED_INCOMES,detail,64535,6143
2,CREDIT_FIXED_INCOMES,balances,5447,504
3,CREDIT_FIXED_INCOMES,detail,5447,504
4,FUNDS,balances,8355,762
5,FUNDS,detail,8355,762
6,TREASURE_TITLES,balances,9138,822
7,TREASURE_TITLES,detail,9138,822
8,VARIABLE_INCOMES,balances,17570,1611
9,VARIABLE_INCOMES,detail,16479,1509


In [14]:
# Verify: within a single snapshot, do we always get both balances and detail for each holding?
con.execute("""
  WITH per_kind AS (
    SELECT snapshot_id, investment_id, payload_kind, count(*) AS n
    FROM pos GROUP BY 1,2,3
  )
  SELECT b.n AS balances_per_snap, d.n AS details_per_snap, count(*) AS occurrences
  FROM (SELECT snapshot_id, investment_id, n FROM per_kind WHERE payload_kind='balances') b
  FULL JOIN (SELECT snapshot_id, investment_id, n FROM per_kind WHERE payload_kind='detail') d
    USING (snapshot_id, investment_id)
  GROUP BY 1,2 ORDER BY occurrences DESC
""").df()

,balances_per_snap,details_per_snap,occurrences
0,1,1.0,103954
1,1,NaN,2739


### 2.5 · The odd columns — `snapshot_id`, `s3_uri`, `page`, `payload_source`

- **`snapshot_id`** — the grouping ID for one sync from one institution for one customer. A snapshot contains N holdings × 2 payloads.
- **`s3_uri`** — provenance: where in blob storage the raw JSON lives. Useful for audit; not for querying.
- **`page`** (transactions only) — Open Finance transactions endpoints paginate; this records which page this row came from.
- **`payload_source`** (transactions only) — which endpoint served the record: `transactions` (historical) or `transactions-current` (recent-only, ~7 days). Different completeness / freshness guarantees.

In [15]:
con.execute("SELECT payload_source, count(*) FROM txn GROUP BY 1").df()

,payload_source,count_star()
0,transactions-current,6432
1,transactions,314685


In [16]:
# Snapshots — one per (institution, customer, sync run)
con.execute("""
  SELECT
    count(DISTINCT snapshot_id) AS snapshots,
    count(*) / count(DISTINCT snapshot_id)::float AS avg_records_per_snapshot
  FROM pos
""").df()

,snapshots,avg_records_per_snapshot
0,5586,37.709808


## 3 · Opening the JSON — position payloads

The payload structure differs per `(investment_type, payload_kind)`. Five families × two kinds = 10 payload shapes. Each cell below picks a real sample and shows it pretty-printed, with per-field notes from the OFB spec.

Helper to pull a sample:

In [17]:
def sample_pos(inv_type: str, kind: str) -> dict:
    row = con.execute(
        "SELECT payload_json FROM pos WHERE investment_type=? AND payload_kind=? LIMIT 1",
        [inv_type, kind]
    ).fetchone()
    return json.loads(row[0])['data']  # unwrap the {data, meta, links} envelope

def sample_txn(inv_type: str) -> dict:
    row = con.execute(
        "SELECT transaction_json FROM txn WHERE investment_type=? LIMIT 1",
        [inv_type]
    ).fetchone()
    return json.loads(row[0])

### 3.1 · `VARIABLE_INCOMES` — equities, ETF, FII

**Spec**: [variable-incomes/1.3.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/variable-incomes/1.3.0.yml)

**detail** — the security's static identity. Very short: ISIN + ticker + issuer CNPJ. This is what makes variable-income the easiest family to derive a logical key for: `(isinCode, ticker)` is highly deterministic — those are B3-standardized.

**balances** — daily snapshot. `closingPrice` is the price on `referenceDate` (which is D-1 or D-2 per the spec).

In [18]:
print('--- detail ---');   print(json.dumps(sample_pos('VARIABLE_INCOMES','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('VARIABLE_INCOMES','balances'), indent=2, ensure_ascii=False))

--- detail ---
{
  "issuerInstitutionCnpjNumber": "60872504000123",
  "isinCode": "BRITUBACNPR1",
  "ticker": "ITUB4"
}

--- balances ---
{
  "referenceDate": "2026-07-30",
  "priceFactor": "1.00",
  "grossAmount": {
    "amount": "161841.19",
    "currency": "BRL"
  },
  "blockedBalance": {
    "amount": "0.00",
    "currency": "BRL"
  },
  "quantity": "6301.00000000",
  "closingPrice": {
    "amount": "25.69",
    "currency": "BRL"
  }
}


### 3.2 · `FUNDS` — investment funds

**Spec**: [funds/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/funds/1.1.0.yml)

**detail** — the fund's CNPJ is the natural logical key (every Brazilian fund has a unique CNPJ). ANBIMA fields are the industry taxonomy for categorizing the fund's strategy.

**balances** — funds are measured in *quotas*, not shares. `quotaGrossPriceValue` is the price of one quota on `referenceDate`. `grossAmount ≈ quotaQuantity × quotaGrossPriceValue` (identity check for us).

In [19]:
print('--- detail ---');   print(json.dumps(sample_pos('FUNDS','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('FUNDS','balances'), indent=2, ensure_ascii=False))

--- detail ---
{
  "name": "SPX NIMITZ ESTRUTURAL FIC FIM",
  "cnpjNumber": "34177667000185",
  "anbimaCategory": "MULTIMERCADO"
}

--- balances ---
{
  "referenceDate": "2026-07-30",
  "grossAmount": {
    "amount": "171027.92",
    "currency": "BRL"
  },
  "netAmount": {
    "amount": "170600.35",
    "currency": "BRL"
  },
  "incomeTaxProvision": {
    "amount": "427.57",
    "currency": "BRL"
  },
  "financialTransactionTaxProvision": {
    "amount": "0.00",
    "currency": "BRL"
  },
  "blockedAmount": {
    "amount": "0.00",
    "currency": "BRL"
  },
  "quotaQuantity": "4472.33766234",
  "quotaGrossPriceValue": {
    "amount": "38.24",
    "currency": "BRL"
  }
}


### 3.3 · `BANK_FIXED_INCOMES` — CDB, LCI, LCA, LC, LF

**Spec**: [bank-fixed-incomes/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/bank-fixed-incomes/1.1.0.yml)

**detail** — richer than variable income. Includes `remuneration` (the interest formula: fixed, indexed to CDI/SELIC/IPCA), `dueDate`, `issueDate`, `clearingCode` (B3 identifier). These are *per-customer* securities (a CDB is issued to you specifically), so the logical key needs several fields: `(isinCode, issuerInstitutionCnpjNumber, dueDate, issueDate)` at minimum.

**balances** — includes `updatedUnitPrice` (marked-to-market unit value) and `purchaseUnitPrice` (what you paid). `netAmount` accounts for accrued income tax (`incomeTax`). Also carries `postFixedIndexerPercentage` — how much of the indexer you get (e.g. 102% of CDI).

In [20]:
print('--- detail ---');   print(json.dumps(sample_pos('BANK_FIXED_INCOMES','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('BANK_FIXED_INCOMES','balances'), indent=2, ensure_ascii=False))

--- detail ---
{
  "issuerInstitutionCnpjNumber": "60746948000112",
  "isinCode": "BRBANKLCA3A3",
  "investmentType": "LCA",
  "remuneration": {
    "rateType": "EXPONENCIAL",
    "ratePeriodicity": "DIARIO",
    "calculation": "DIAS_UTEIS",
    "indexer": "SELIC",
    "postFixedIndexerPercentage": "1.020000"
  },
  "issueUnitPrice": {
    "amount": "1000.00",
    "currency": "BRL"
  },
  "dueDate": "2024-04-01",
  "issueDate": "2021-04-05",
  "clearingCode": "LCA60746948",
  "purchaseDate": "2021-04-05",
  "gracePeriodDate": "2024-04-01"
}

--- balances ---
{
  "referenceDateTime": "2026-07-30T09:00:00Z",
  "quantity": "18.96700000",
  "updatedUnitPrice": {
    "amount": "991.50",
    "currency": "BRL"
  },
  "grossAmount": {
    "amount": "18805.78",
    "currency": "BRL"
  },
  "netAmount": {
    "amount": "18758.77",
    "currency": "BRL"
  },
  "incomeTax": {
    "amount": "47.01",
    "currency": "BRL"
  },
  "financialTransactionTax": {
    "amount": "0.00",
    "currency": "BRL

### 3.4 · `CREDIT_FIXED_INCOMES` — CRI, CRA, Debêntures

**Spec**: [credit-fixed-incomes/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/credit-fixed-incomes/1.1.0.yml)

Shape mirrors bank-fixed but adds `debtorCnpjNumber` / `debtorName` — the *ultimate* borrower (for a debênture, the company; for a CRI, the securitizer). Logical key candidate: `(isinCode, debtorCnpjNumber, dueDate)`.

**Tier-1 defect to watch**: sample below likely shows `issuerInstitutionCnpjNumber` ending in `.00` — that's the *tax-id decimal tail* from the case study, appearing in real synthetic data. This is what the raw contract will need to coerce.

In [21]:
print('--- detail ---');   print(json.dumps(sample_pos('CREDIT_FIXED_INCOMES','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('CREDIT_FIXED_INCOMES','balances'), indent=2, ensure_ascii=False))

--- detail ---
{
  "issuerInstitutionCnpjNumber": "92894922000108.00",
  "isinCode": "BRDEBSEC01A1",
  "investmentType": "DEBENTURES",
  "debtorCnpjNumber": "09149503000106",
  "debtorName": "OMEGA ENERGIA S.A.",
  "taxExemptProduct": "NAO",
  "remuneration": {
    "rateType": "EXPONENCIAL",
    "ratePeriodicity": "ANUAL",
    "calculation": "DIAS_UTEIS",
    "indexer": "CDI",
    "postFixedIndexerPercentage": "1.020000"
  },
  "issueUnitPrice": {
    "amount": "1000.00",
    "currency": "BRL"
  },
  "issueDate": "2022-08-06",
  "dueDate": "2029-06-15",
  "voucherPaymentIndicator": "SIM",
  "voucherPaymentPeriodicity": "SEMESTRAL",
  "clearingCode": "DEB01091495",
  "purchaseDate": "2023-03-04"
}

--- balances ---
{
  "referenceDateTime": "2026-07-30T09:00:00Z",
  "updatedUnitPrice": {
    "amount": "989.24",
    "currency": "BRL"
  },
  "quantity": "248.67300000",
  "grossAmount": {
    "amount": "245996.28",
    "currency": "BRL"
  },
  "netAmount": {
    "amount": "245381.29",
    "

### 3.5 · `TREASURE_TITLES` — Tesouro Direto (government bonds)

**Spec**: [treasure-titles/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/treasure-titles/1.1.0.yml)

**detail** — standardized government product. `isinCode` alone is a strong logical key (every Tesouro title has a unique national ISIN). `productName` is human-readable ("Tesouro Prefixado 2027", "Tesouro IPCA+ 2035", ...).

**balances** — `updatedUnitPrice` reflects the bond's marked-to-market value (moves with rates). `purchaseUnitPrice` is what the customer paid. Difference × `quantity` ≈ unrealized gain.

In [22]:
print('--- detail ---');   print(json.dumps(sample_pos('TREASURE_TITLES','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('TREASURE_TITLES','balances'), indent=2, ensure_ascii=False))

--- detail ---
{
  "isinCode": "BRSTNCLTN8C3",
  "productName": "Tesouro Prefixado 2027",
  "remuneration": {
    "indexer": "PRE_FIXADO",
    "ratePeriodicity": "ANUAL",
    "calculation": "DIAS_UTEIS",
    "preFixedRate": "0.112500"
  },
  "dueDate": "2027-01-01",
  "purchaseDate": "2026-03-12",
  "voucherPaymentIndicator": "NAO"
}

--- balances ---
{
  "referenceDateTime": "2026-07-30T09:00:00Z",
  "updatedUnitPrice": {
    "amount": "504.63",
    "currency": "BRL"
  },
  "grossAmount": {
    "amount": "110316.62",
    "currency": "BRL"
  },
  "netAmount": {
    "amount": "110040.83",
    "currency": "BRL"
  },
  "incomeTax": {
    "amount": "275.79",
    "currency": "BRL"
  },
  "financialTransactionTax": {
    "amount": "0.00",
    "currency": "BRL"
  },
  "blockedBalance": {
    "amount": "0.00",
    "currency": "BRL"
  },
  "purchaseUnitPrice": {
    "amount": "500.00",
    "currency": "BRL"
  },
  "quantity": "218.61000000"
}


## 4 · Opening the JSON — transaction payloads

Transactions are pre-flattened: one row per movement, not one row per API page. All five families share a broadly similar shape (`type`, `transactionType`, `transactionDate`, `transactionQuantity`, `transactionUnitPrice`, `transactionValue` / `transactionNetValue`, `transactionId`), with per-family additions.

Note the enums:
- **`type`** — direction: `ENTRADA` (money/quantity in) / `SAIDA` (out). Universal across families.
- **`transactionType`** — the semantic action: `COMPRA`, `APLICACAO`, `RESGATE`, `RENDIMENTO`, `JUROS`, `AMORTIZACAO`, ... Per-family enum in `EnumXxxTransactionsTransactionType`.

**Consumption layer implication**: the wealth page needs a *unified* type — user-facing categories like `buy`, `sell`, `income`, `transfer`. That mapping table lives in the consumption layer (or a shared contract), not in canonical.

In [23]:
for it in ['VARIABLE_INCOMES','FUNDS','BANK_FIXED_INCOMES','CREDIT_FIXED_INCOMES','TREASURE_TITLES']:
    print(f'### {it} ###')
    print(json.dumps(sample_txn(it), indent=2, ensure_ascii=False))
    print()

### VARIABLE_INCOMES ###
{
  "type": "ENTRADA",
  "transactionType": "COMPRA",
  "transactionDate": "2020-04-06",
  "priceFactor": "1.00000000",
  "transactionQuantity": "6301.00000000",
  "transactionUnitPrice": {
    "amount": "25.00000000",
    "currency": "BRL"
  },
  "transactionValue": {
    "amount": "157525.0000",
    "currency": "BRL"
  },
  "brokerNoteId": "75644578",
  "transactionId": "826e8d3e-1745-5e01-a161-c478a02917b8"
}

### FUNDS ###
{
  "transactionId": "0795301975648094",
  "type": "ENTRADA",
  "transactionType": "APLICACAO",
  "transactionConversionDate": "2020-06-10",
  "transactionQuotaPrice": {
    "amount": "38.50000000",
    "currency": "BRL"
  },
  "transactionQuotaQuantity": "4472.33766234",
  "transactionValue": {
    "amount": "172185.0000",
    "currency": "BRL"
  },
  "transactionGrossValue": {
    "amount": "172185.0000",
    "currency": "BRL"
  }
}

### BANK_FIXED_INCOMES ###
{
  "type": "ENTRADA",
  "transactionType": "APLICACAO",
  "transactionDate":

In [24]:
# What transactionType values actually appear per family?
con.execute("""
  SELECT investment_type,
         json_extract_string(transaction_json, '$.transactionType') AS tx_type,
         count(*) AS n
  FROM txn
  GROUP BY 1,2 ORDER BY 1, 3 DESC
""").df()

,investment_type,tx_type,n
0,BANK_FIXED_INCOMES,APLICACAO,87517
1,BANK_FIXED_INCOMES,RESGATE,75233
2,BANK_FIXED_INCOMES,OUTROS,20215
3,BANK_FIXED_INCOMES,PAGAMENTO_JUROS,8928
4,BANK_FIXED_INCOMES,VENCIMENTO,2407
5,BANK_FIXED_INCOMES,TRANSFERENCIA_CUSTODIA,1473
6,BANK_FIXED_INCOMES,AMORTIZACAO,1376
7,BANK_FIXED_INCOMES,TRANSFERENCIA_TITULARIDADE,447
8,CREDIT_FIXED_INCOMES,COMPRA,9785
9,CREDIT_FIXED_INCOMES,VENDA,2343


## 5 · A first pass at the defects

Now that we know the actual field names, wire up the three named defect probes. Keep results here — feeds directly into `design/data_quality.md`.

### 5.1 · Tier-1 · Tax-id decimal tail (from the case study, named)

Tax IDs (CNPJ, CPF) are 14 or 11 digit strings. Any value with `.` in it is the exact defect the case study warns about.

In [25]:
con.execute("""
  SELECT investment_type,
         count(*) FILTER (WHERE json_extract_string(payload_json, '$.data.issuerInstitutionCnpjNumber') LIKE '%.%') AS decimal_tail_issuer,
         count(*) FILTER (WHERE json_extract_string(payload_json, '$.data.debtorCnpjNumber') LIKE '%.%')          AS decimal_tail_debtor,
         count(*) AS total
  FROM pos
  WHERE payload_kind = 'detail'
  GROUP BY 1 ORDER BY 1
""").df()

,investment_type,decimal_tail_issuer,decimal_tail_debtor,total
0,BANK_FIXED_INCOMES,1370,0,64535
1,CREDIT_FIXED_INCOMES,306,0,5447
2,FUNDS,0,0,8355
3,TREASURE_TITLES,0,0,9138
4,VARIABLE_INCOMES,0,0,16479


### 5.2 · Tier-2 · Intra-sync duplicate

For variable-income (easiest logical key), check: same `(snapshot_id, account_id, isinCode, ticker)` under >1 `investment_id`.

In [26]:
con.execute("""
  WITH d AS (
    SELECT snapshot_id, account_id, investment_id,
           json_extract_string(payload_json, '$.data.isinCode') AS isin,
           json_extract_string(payload_json, '$.data.ticker')   AS ticker
    FROM pos
    WHERE investment_type = 'VARIABLE_INCOMES' AND payload_kind = 'detail'
  )
  SELECT snapshot_id, account_id, isin, ticker,
         count(DISTINCT investment_id) AS n_ids,
         array_agg(DISTINCT investment_id) AS ids
  FROM d
  GROUP BY 1,2,3,4
  HAVING count(DISTINCT investment_id) > 1
  ORDER BY n_ids DESC LIMIT 20
""").df()

,snapshot_id,account_id,isin,ticker,n_ids,ids
0,ba69c129-75cb-5c1d-8160-e0eca6d32060,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6420e29e-a891-5cb9-859e-696da03e7a31, 7..."
1,39dff561-4127-5b01-9c14-eb1d28c1dbc4,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6420e29e-a891-5cb9-859e-696da03e7a31, 7..."
2,141bad3d-6efc-52bb-9287-2d40c570bb9f,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[3758c1da-6799-507a-acd8-2d6c9f65cfdc, f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6..."
3,587debfe-73f7-5d8c-ac73-0563778ec4d6,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6420e29e-a891-5cb9-859e-696da03e7a31, 7..."
4,77895887-b582-570c-9ded-0923346c4ff3,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6420e29e-a891-5cb9-859e-696da03e7a31, 7..."
5,4c01193b-999d-5435-8fee-d3470cd7670c,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[3758c1da-6799-507a-acd8-2d6c9f65cfdc, f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6..."
6,c24c41cc-fded-5da5-9a71-d0aa294c91f6,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[3758c1da-6799-507a-acd8-2d6c9f65cfdc, f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6..."
7,1325f123-ecdf-5a34-88d9-f389cca9c98f,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6420e29e-a891-5cb9-859e-696da03e7a31, 7..."
8,c24c41cc-fded-5da5-9a71-d0aa294c91f6,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRB3SAACNOR6,B3SA3,4,"[9fe25af5-2b4c-5ee3-93fa-2b93475ea9f3, 09d9db12-8b74-5d71-a304-0b995f846112, 970ab59c-aa0e-520a-b9da-c9edac78838a, a..."
9,6c9ef166-2363-5b18-ad7c-2d4b64ecaa85,6b8fcd56-8159-51fd-820d-8aa84292931c,BRPETRACNPR6,PETR4,4,"[caefa263-9034-57a5-b6d6-824196dcdf8b, e697b500-1ab0-59d1-88f7-55f70e452d2b, d9fc51e1-18c7-55fe-b6c6-b6b2b220bd74, 8..."


### 5.3 · Tier-2 · Identity churn

Same logical key across snapshots under >1 `investment_id`. If any hits appear, the institution is violating the OFB spec's `investmentId` reuse mandate.

In [27]:
con.execute("""
  WITH d AS (
    SELECT institution_id, account_id, investment_id,
           json_extract_string(payload_json, '$.data.isinCode') AS isin,
           json_extract_string(payload_json, '$.data.ticker')   AS ticker
    FROM pos
    WHERE investment_type = 'VARIABLE_INCOMES' AND payload_kind = 'detail'
  )
  SELECT institution_id, account_id, isin, ticker,
         count(DISTINCT investment_id) AS n_ids,
         array_agg(DISTINCT investment_id) AS ids
  FROM d
  GROUP BY 1,2,3,4
  HAVING count(DISTINCT investment_id) > 1
  ORDER BY n_ids DESC LIMIT 20
""").df()

,institution_id,account_id,isin,ticker,n_ids,ids
0,00000000000006,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[3758c1da-6799-507a-acd8-2d6c9f65cfdc, f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6..."
1,00000000000006,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRVALEACNOR0,VALE3,4,"[b356be12-248c-5f24-8d9d-c69518e47a08, 88bebc82-9713-53fa-be23-9c615abe290b, 8bb8e97c-6970-558e-95af-5dcfd3e78458, 8..."
2,00000000000006,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRBPACUNT002,BPAC11,4,"[0b42c9ef-019b-58ac-8dc0-3bd0a9e9ad60, 12ed56f3-cf2d-5953-8142-d8753f20d1f6, bb30bb64-2206-530c-ba48-f943ee3e2851, 0..."
3,00000000000001,6b8fcd56-8159-51fd-820d-8aa84292931c,BRHASHCTF001,HASH11,4,"[a48b86d8-75e5-5980-bf82-4ae35e1e869f, ec34df5a-d547-5bab-b6f4-4eb7722d6fc2, 4a78e558-78e4-5a9b-a879-a1ddc2eed7ca, 3..."
4,00000000000002,2ab892a8-1694-5b17-a13f-eb758d48210c,BRWEGEACNOR0,WEGE3,4,"[6073627565773498, 4771988618428728, 0742083929654326, 4159843754026982]"
5,00000000000004,ccdc8073-a0e3-5799-992d-9af6310ba197,BRBPACUNT002,BPAC11,4,"[5352657126479291, 4089541970803799, 1185747893514315, 9933514857547071]"
6,00000000000004,ccdc8073-a0e3-5799-992d-9af6310ba197,BRITUBACNPR1,ITUB4,4,"[7317469540200850, 2780314525183112, 2629220036651403, 0542240753178469]"
7,00000000000002,2ab892a8-1694-5b17-a13f-eb758d48210c,BRHASHCTF001,HASH11,4,"[6941015890155257, 5908521798068715, 3589241947719036, 1183330148925721]"
8,00000000000006,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRBBDCACNPR8,BBDC4,4,"[ee821b08-741b-5c35-b9ea-514d61da6fd6, aa67f8d6-d16a-542f-91ef-7159de1e53a7, f1aeba24-2979-5670-9d9c-28232ec62410, 3..."
9,00000000000006,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRB3SAACNOR6,B3SA3,4,"[ae1bcec5-8561-5db3-a5d9-15d994570271, 9fe25af5-2b4c-5ee3-93fa-2b93475ea9f3, 09d9db12-8b74-5d71-a304-0b995f846112, 9..."


### 5.4 · Tier-2 · Zero-flap

Balance = 0 sandwiched between non-zero, quantity unchanged.

In [28]:
con.execute("""
  WITH b AS (
    SELECT party_id, investment_id, snapshot_created_at,
           cast(json_extract_string(payload_json, '$.data.quantity') AS DOUBLE)                 AS qty,
           cast(json_extract_string(payload_json, '$.data.grossAmount.amount') AS DOUBLE)      AS gross
    FROM pos WHERE payload_kind = 'balances'
  ),
  o AS (
    SELECT *,
           lag(gross)  OVER w AS prev_v, lead(gross) OVER w AS next_v,
           lag(qty)    OVER w AS prev_q, lead(qty)   OVER w AS next_q
    FROM b
    WINDOW w AS (PARTITION BY party_id, investment_id ORDER BY snapshot_created_at)
  )
  SELECT party_id, investment_id, snapshot_created_at, qty, gross, prev_v, next_v
  FROM o
  WHERE gross = 0 AND prev_v > 0 AND next_v > 0 AND qty = prev_q AND qty = next_q
  ORDER BY snapshot_created_at LIMIT 20
""").df()

,party_id,investment_id,snapshot_created_at,qty,gross,prev_v,next_v
0,679aa92e-f12b-5860-9d15-8a2838bf3e3f,5667609725231687,2026-08-06T10:04:11Z,5376.0,0.0,212734.77,216209.82
1,679aa92e-f12b-5860-9d15-8a2838bf3e3f,2841986982285407,2026-08-06T10:04:11Z,1589.0,0.0,75199.74,76177.93
2,5a5c2d94-f9d9-5287-9baa-1b25e0030872,9596c2ee-3b47-5dc9-a6fd-ff0c9688070b,2026-08-07T09:22:31Z,5778.0,0.0,227616.22,229881.20
3,43037b26-0d10-577f-b627-b17d7bdd4c30,4880537593578090,2026-08-07T09:58:31Z,2059.0,0.0,110051.49,115007.09
4,43037b26-0d10-577f-b627-b17d7bdd4c30,2351446269028886,2026-08-07T09:58:31Z,118.0,0.0,1183.35,1222.01
5,43037b26-0d10-577f-b627-b17d7bdd4c30,6192164993522306,2026-08-07T09:58:31Z,3321.0,0.0,204009.03,208691.64
6,e8cc5faa-568f-5cbd-b7e1-d69e3d4c889e,5203732272454177,2026-08-08T09:16:49Z,4879.0,0.0,117749.79,126258.76
7,c0a77995-138b-5df2-a681-957af5137b34,fc7fa1be-45a8-5d8a-afdc-aaf959501c2d,2026-08-08T09:21:30Z,2971.0,0.0,168339.83,165601.16
8,660e7b49-d0a8-57e4-950a-8b94003b0cdf,c7c2772d-5990-5310-807a-3e4e656e8710,2026-08-08T10:24:21Z,8217.0,0.0,141916.63,139334.03
9,7dbd3a98-3340-5e34-ae5b-e2e8806a0052,6515843439509224,2026-08-09T09:14:37Z,4887.0,0.0,160441.19,161838.87


## 6 · Scratch-pad — observations for the design docs

Fill this in as you go. These bullets are what will become `design/decisions.md` and `design/data_quality.md`.

- **Grain of positions**: one snapshot × one holding = **two rows** (balances + detail). Canonical `positions` should be the join of the two, keyed on `(party_id, account_id, snapshot_id, investment_id)` at natural grain.
- **Watermark**: `ingested_at` on both files. `snapshot_created_at` is the event-time; the gap between them is the pipeline lag.
- **Logical key** per family — record what actually works when you probed above:
  - `VARIABLE_INCOMES`: `(isinCode, ticker)` — check that intra-sync dup query returned 0
  - `FUNDS`: `(cnpjNumber)` — write and run
  - `BANK_FIXED_INCOMES`: `(isinCode, issuerInstitutionCnpjNumber, dueDate, issueDate)` — write and run
  - `CREDIT_FIXED_INCOMES`: `(isinCode, debtorCnpjNumber, dueDate)` — write and run
  - `TREASURE_TITLES`: `(isinCode)` — write and run
- **Tier-1 defects observed**: fill in — decimal-tail count, missing-required count, enum violations.
- **Tier-2 defects observed**: fill in — intra-sync dup count per family, identity churn count per family, zero-flap count.
- **Transaction type mapping**: enumerate distinct values seen in the query above; group them into wealth-consumer categories (`buy`, `sell`, `income`, `transfer`, `tax`, `other`).